## Simulate GPT OSS 120B as participant in code switching

### Import and process the items

In [1]:
import pandas as pd
from pathlib import Path
from openai import OpenAI
import concurrent.futures


BASE_DIR = Path.cwd().parent
INPUT  = BASE_DIR / "Human_data" / "coding_clean_final.xlsx"
OUTPUT = BASE_DIR / "LLM_data" 

df_human = pd.read_excel(INPUT)

def build_participant_items(df: pd.DataFrame) -> dict:
    participant_items = {}
    for pid, group in df.groupby("ID", sort=False):
        participant_items[pid] = group[["Item", "Itemtype"]] \
                                      .to_dict(orient="records")
    return participant_items

participant_items = build_participant_items(df_human)

### Request LLM

No available adjustment for temperature and top-P

GPT-OSS-120B is a reasonging model, we can only adjust reasoning ability. 

So I use "low" "medium" "high" to test how this parameter affect the results.

In [2]:
# initial setup
MODEL_NAME = "gpt-oss:120b"

client = OpenAI(
    base_url="http://localhost:11434/v1", # base url change to server url via Ollama interface
    api_key="ollama", # change api key to ollama
    timeout=120,
    max_retries=3,
)

# system prompt
SYSTEM_PROMPT = """You are a Spanish-English bilingual. You are participating in a linguistic task.

In this study, you will read sentences in English, Spanish or a mixture of Spanish and English. This study is completing sentences.

You will see sentences that are missing the last word. You are asked to complete each sentence in a plausible way with the first word that comes to mind. You are asked to complete only one word.

There are three sentence types. The following is an example of an English sentence. Here you would need to complete an English word.

Martha wants to buy groceries and goes to the ...

A possible completion could be the following:
Martha wants to buy groceries and goes to the ...
supermarket.

Una oración también puede estar completamente en español. A continuación, se muestra un ejemplo. Aquí, se escribe una palabra en español para completar la oración:
Marta quiere hacer compras y va al ...
supermercado.

Una oración también puede empezar in Spanish and change to English, as in the following example. Here you would complete an English word:
Marta quiere hacer compras and goes to the ...
supermarket.

Please write down what first comes to mind. Please complete the sentence with only one word. Please complete a Spanish sentence with a Spanish word, and the English and mixed sentences with an English word. Your completion needs to be plausible and grammatically correct, but other than that, you can write anything.

Important: only return the word."""

In [3]:
def run_single_trial(participant_id: str, item: dict) -> dict:
    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            reasoning_effort="low", # we choose low reasoning effort because we found low is the best setting for reflecting human behavior
            temperature = 0.2, # we choose temperature as [0, 0.2, 0.4, 0.6, 0.8, 1] to test the effect of temperature on the model's performance.
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": item["Item"]}
            ]
        )

        choice = response.choices[0]
        usage = response.usage

        return {
            "ID": participant_id,
            "Item": item["Item"],
            "Itemtype": item["Itemtype"],

            # actual final answer
            "Completion": (
                choice.message.content.strip()
                if choice.message.content
                else None
            ),

            # reasoning text
            "Reasoning": (
                choice.message.reasoning
                if getattr(choice.message, "reasoning", None)
                else None
            ),

            # generation information
            "finish_reason": choice.finish_reason,

            # token statistics
            "prompt_tokens": (
                usage.prompt_tokens
                if usage else None
            ),
            "completion_tokens": (
                usage.completion_tokens
                if usage else None
            ),
            "total_tokens": (
                usage.total_tokens
                if usage else None
            ),

            # Ollama does not currently expose this
            "reasoning_tokens": None,

            "status": "success"
        }

    except Exception as e:
        return {
            "ID": participant_id,
            "Item": item["Item"],
            "Itemtype": item["Itemtype"],
            "Completion": None,
            "Reasoning": None,
            "finish_reason": None,
            "prompt_tokens": None,
            "completion_tokens": None,
            "total_tokens": None,
            "reasoning_tokens": None,
            "status": str(e)
        }


def run_participant(participant_id: str, items: list[dict], max_workers: int = 3) -> list[dict]:
    with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(run_single_trial, participant_id, item)
            for item in items
        ]
        return [f.result() for f in concurrent.futures.as_completed(futures)]

def run_all(participant_items: dict, max_workers: int = 5) -> pd.DataFrame:
    all_results = []
    participants = list(participant_items.items())
    total = len(participants)

    for i, (pid, items) in enumerate(participants, 1):
        print(f"Running participant {i}/{total}: {pid}")

        results = run_participant(pid, items, max_workers)
        all_results.extend(results)

        if i % 10 == 0:
            df_checkpoint = pd.DataFrame(all_results)
            checkpoint_path = OUTPUT / f"checkpoint_{i}of{total}.csv"
            df_checkpoint.to_csv(checkpoint_path, index=False)
            print(f"  ✓ Checkpoint: {checkpoint_path.name}")

    return pd.DataFrame(all_results)

In [4]:
# Run all participants and get the results in a DataFrame
df_llm = run_all(participant_items)

Running participant 1/120: 1750873464
Running participant 2/120: 1750874407
Running participant 3/120: 1750875716
Running participant 4/120: 1750875995
Running participant 5/120: 1750879780
Running participant 6/120: 1750887668
Running participant 7/120: 1751968832
Running participant 8/120: 1751969612
Running participant 9/120: 1751969748
Running participant 10/120: 1751970033
  ✓ Checkpoint: checkpoint_10of120.csv
Running participant 11/120: 1751971669
Running participant 12/120: 1751973483
Running participant 13/120: 1751974229
Running participant 14/120: 1751977115
Running participant 15/120: 1751978691
Running participant 16/120: 1751978981
Running participant 17/120: 1751980278
Running participant 18/120: 1751980488
Running participant 19/120: 1751981228
Running participant 20/120: 1751981332
  ✓ Checkpoint: checkpoint_20of120.csv
Running participant 21/120: 1751981512
Running participant 22/120: 1751983542
Running participant 23/120: 1751986873
Running participant 24/120: 175198

### Check and save the output data

In [8]:
df_llm.head(10)

,ID,Item,Itemtype,Completion,Reasoning,finish_reason,prompt_tokens,completion_tokens,total_tokens,reasoning_tokens,status
0,1750873464,Marsha knew the guests could remain eating app...,English,hour,We need one word continuation in English. Sent...,stop,377,45,422,None,success
1,1750873464,Everyone was honking at the old man who was dr...,English,slowly,We need one word continuation in English. Sent...,stop,376,59,435,None,success
2,1750873464,The boss of the store where he worked afternoo...,English,raise,We need one word continuation in English (sent...,stop,378,52,430,None,success
3,1750873464,Die junge Mutter aus der Nachbarschaft füttert...,code_switching,milk,We need one word in English because sentence i...,stop,379,45,424,None,success
4,1750873464,The man downstairs played his stereo much too,English,loud,We just need one word continuation in English ...,stop,372,39,411,None,success
5,1750873464,"Der Lehrer sah, dass einige Schüler schummelte...",code_switching,answers,We need one word in English because mixed sent...,stop,378,51,429,None,success
6,1750873464,Man kann die Tür nicht öffnen with the wrong,code_switching,key,We need one word completing sentence. It's Ger...,stop,373,53,426,None,success
7,1750873464,She had to leave the movie when the baby start...,English,cry,"We just output one word, likely ""cry"". English...",stop,375,28,403,None,success
8,1750873464,Most people in our country spend December 25th...,English,family,We need one word completing sentence. It's Eng...,stop,376,43,419,None,success
9,1750873464,"Giraffen sind für ihre langen Hälse bekannt, a...",code_switching,stripes,We need one word in English because mixed sent...,stop,382,31,413,None,success


In [10]:
# check failed trial
failed = df_llm[df_llm["status"] != "success"]
if len(failed) > 0:
   print(f"⚠️ failed trial: {len(failed)}个")
print(failed[["ID", "Item", "status"]])
else:
print(f"✓ all success，in total of {len(df_llm)} trials")
# delete status column and save to excel
#df_llm = df_llm.drop(columns=["status"])
#df_llm.to_csv(OUTPUT / f"llm_{MODEL_NAME.replace(':', '-')}_r_low_t2_2.csv", index=False)

SyntaxError: invalid syntax (891743264.py, line 6)

### test

In [67]:
# ── use the first participant for testing ─────────────────────────────────────
first_pid = list(participant_items.keys())[0]
first_items = participant_items[first_pid]

for item in first_items[:3]:
    print(f"  {item}")

# ── only run this participant ───────────────────────────────────────
#test single participant
test_results = run_participant(first_pid, first_items)
df_test = pd.DataFrame(test_results)

print(
    df_test[
        [
            "Itemtype",
            "Completion",
            "finish_reason",
            "completion_tokens",
            "reasoning_tokens",
            "status"
        ]
    ].to_string(index=False)
)

  {'Item': 'Everyone was honking at the old man who was driving too', 'Itemtype': 'English'}
  {'Item': 'Die junge Mutter aus der Nachbarschaft fütterte her baby some warm', 'Itemtype': 'code_switching'}
  {'Item': 'The man downstairs played his stereo much too', 'Itemtype': 'English'}
      Itemtype Completion finish_reason  completion_tokens reasoning_tokens  status
       English       loud          stop                 39             None success
       English     slowly          stop                 59             None success
code_switching       milk          stop                 45             None success
       English      raise          stop                 54             None success
       English       hour          stop                 45             None success
code_switching    answers          stop                 50             None success
code_switching        key          stop                 73             None success
       English        cry          stop  